# Soal 2b · Analisis dengan Operasi RDD — **4 driver berbagi 1 cluster**
**Big Data** · dijalankan di **JupyterLab PC Master**; pasangannya **Soal2_Master_Monitor.ipynb** (1 monitor untuk semua driver).

Setiap anggota membuka **salinan** notebook ini (mis. `Soal2b_RDD_driver1.ipynb`) dan hanya mengubah `NOMOR_DRIVER` di Sel 1.
Cluster 16 core dibagi rata: setiap driver mendapat **4 core = 1 core di masing-masing dari 4 PC Worker**, sehingga setiap driver tetap tersebar ke 4 PC.

| Driver | Mengerjakan | Spark UI |
|---|---|---|
| 1 | (i) 10 wiki dengan edit manusia terbanyak | :4040 |
| 2 | (ii) 10 halaman idwiki paling banyak disunting | :4041 |
| 3 | (iii) 15 kata terbanyak pada comment idwiki | :4042 |
| 4 | (i), (ii), (iii) sebagai **pembanding** (Sel 9 mencocokkan hasilnya dengan driver 1–3) | :4043 |

| Operasi | Jenis | Alasan |
|---|---|---|
| `filter`, `map`, `flatMap` | **narrow** | partisi hasil hanya bergantung pada satu partisi asal; tidak ada data berpindah antar-PC |
| `reduceByKey` | **wide** | kunci yang sama tersebar di banyak partisi → *shuffle*; Spark memulai stage baru |
| `takeOrdered` | **action** | memicu eksekusi; N teratas per partisi digabung di driver, tanpa shuffle |

**Sebelum mulai:** matikan driver lain di luar keempat driver ini (mis. `Soal2_RDD.ipynb`); di monitor jalankan Sel 1, Sel 4 (kosongkan), Sel 3.
Urutan notebook ini: **Sel 1 → 8** (Sel 9 hanya untuk driver 4, setelah driver 1–3 selesai).
Waktu di sini **termasuk jeda `pantau()`** dan dibagi dengan driver lain, jadi **tidak** dipakai untuk perbandingan performa (Soal 2c/2d memakai 1 driver 16 core).

## Sel 1. SparkSession ke cluster, `tandai()`, `report()`, `pantau()`

In [1]:
# ===== SEL 1: SparkSession, tandai, report, pantau =====
import os, sys, socket, time, json, uuid, urllib.request
from pathlib import Path
from operator import add
from pyspark import SparkContext
from pyspark.sql import SparkSession

# ---------------- ISI SESUAI ANGGOTA ----------------
NOMOR_DRIVER   = 2                     # 1, 2, 3, atau 4 (lihat tabel di atas)
# ----------------------------------------------------
MASTER_IP      = "192.168.1.15"
CORES_MAX      = 4                     # 16 core cluster dibagi 4 driver
EXECUTOR_CORES = 1                     # 4 executor x 1 core -> 1 core di tiap PC Worker
EXECUTOR_MEM   = "640m"                # 4 executor harus muat di Worker terkecil (3 GB)
MIN_PARTISI    = 2 * CORES_MAX         # 8 task di tahap 0
N_REDUCE       = CORES_MAX             # 4 task di tahap 1 (setelah reduceByKey)
PERTANYAAN     = {1: {"i"}, 2: {"ii"}, 3: {"iii"}, 4: {"i", "ii", "iii"}}[NOMOR_DRIVER]
D              = f"D{NOMOR_DRIVER}"    # awalan label di monitor dan Spark UI
JEDA_PANTAU    = 1                     # detik per task agar RUNNING terlihat di monitor
DATA_DIR       = "C:/BigData/data/snapshot_wiki"
# -------------------------------------------------------

MASTER     = f"spark://{MASTER_IP}:7077"
REPORT_URL = f"http://{MASTER_IP}:8090"
PATH_ASLI  = f"file:///{DATA_DIR}/wiki_snapshot.jsonl"

s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
s.connect((MASTER_IP, 7077)); IP = s.getsockname()[0]; s.close()
EVENT_DIR = Path("spark-events").resolve(); EVENT_DIR.mkdir(exist_ok=True)

old = SparkContext._active_spark_context
if old is not None:
    try: old.stop()
    except Exception: pass
SparkSession._instantiatedSession = None
SparkSession._activeSession = None

spark = (SparkSession.builder
         .appName(f"Soal2b-driver{NOMOR_DRIVER}")
         .master(MASTER)
         .config("spark.driver.host", IP)
         .config("spark.driver.bindAddress", IP)
         .config("spark.cores.max", str(CORES_MAX))
         .config("spark.executor.cores", str(EXECUTOR_CORES))
         .config("spark.executor.memory", EXECUTOR_MEM)
         .config("spark.pyspark.python", sys.executable)
         .config("spark.pyspark.driver.python", sys.executable)
         .config("spark.ui.port", str(4039 + NOMOR_DRIVER))
         .config("spark.eventLog.enabled", "true")
         .config("spark.eventLog.dir", EVENT_DIR.as_uri())
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

def tandai(deskripsi):
    sc.setJobGroup(deskripsi.split()[0], deskripsi)

def report(run, app, nomor, status):
    """Dijalankan di executor: kirim status task ke monitor (Soal2_Master_Monitor.ipynb)."""
    import json, os, socket, urllib.request
    data = {"run": run, "app": app, "task": nomor, "worker": socket.gethostname(),
            "pid": os.getpid(), "status": status}
    req = urllib.request.Request(REPORT_URL, data=json.dumps(data).encode("utf-8"),
                                 headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=5) as r:
            return r.status == 200
    except Exception as e:
        print(f"Gagal melapor ke monitor: {e}")
        return False

def pantau(rdd, label, jeda=JEDA_PANTAU):
    """Bungkus RDD: tiap partisi (= task) melapor RUNNING lalu DONE (pola materi)."""
    run = str(uuid.uuid4())
    def proses_partition(idx, isi):
        import time
        report(run, label, idx, "RUNNING")
        data = list(isi)
        time.sleep(jeda)
        report(run, label, idx, "DONE")
        return iter(data)
    return rdd.mapPartitionsWithIndex(proses_partition)

print(f"Driver {NOMOR_DRIVER} | mengerjakan {sorted(PERTANYAAN)} | {CORES_MAX} core, executor {EXECUTOR_CORES} core x {EXECUTOR_MEM}")
print("Spark   :", spark.version, "|", sc.master, "| driver host", IP)
print("Spark UI:", sc.uiWebUrl, "-> tab Jobs (label 2b-...) -> DAG Visualization")
try:
    with urllib.request.urlopen(REPORT_URL, timeout=3) as r:
        print("Monitor :", r.read().decode().strip())
except Exception as e:
    print("PERINGATAN: monitor tidak terjangkau ->", e)

Driver 2 | mengerjakan ['ii'] | 4 core, executor 1 core x 640m
Spark   : 4.1.3 | spark://192.168.1.15:7077 | driver host 192.168.1.15
Spark UI: http://192.168.1.15:4041 -> tab Jobs (label 2b-...) -> DAG Visualization
Monitor : Monitor aktif. Jumlah laporan: 7


## Sel 2. Membaca dan mem-parsing snapshot menjadi tuple
Aturan parsing sama dengan Soal 2a. Baris rusak tetap dihitung accumulator (snapshot asli: harus 0).

In [2]:
# ===== SEL 2: textFile + parsing ke tuple =====
WIKI, ID, TYPE, TITLE, USER, IS_ANON, BOT, MINOR, TS, LEN_OLD, LEN_NEW, COMMENT = range(12)
acc_rusak = sc.accumulator(0)

def _int(x):
    return isinstance(x, int) and not isinstance(x, bool)

def parse(baris):
    import json
    try:
        e = json.loads(baris) if baris.strip() else None
    except ValueError:
        e = None
    if not isinstance(e, dict) or not (
            _int(e.get("id")) and _int(e.get("timestamp"))
            and all(isinstance(e.get(k), str) and e.get(k) for k in ("wiki", "type", "title"))):
        acc_rusak.add(1); return []
    return [(e["wiki"], e["id"], e["type"], e["title"], e.get("user"),
             bool(e.get("is_anon", False)), bool(e.get("bot", False)), bool(e.get("minor", False)),
             e["timestamp"], e.get("length_old"), e.get("length_new"), e.get("comment") or "")]

parsed = sc.textFile(PATH_ASLI, minPartitions=MIN_PARTISI).flatMap(parse).setName("parsed")   # narrow
print("Partisi parsed:", parsed.getNumPartitions())

Partisi parsed: 8


## Sel 3. Stopword (broadcast) dan tokenisasi untuk (iii)

In [3]:
# ===== SEL 3: Stopword dan tokenisasi =====
import re
STOPWORD_ID = """
ada adalah agar akan aku anda antara apa apakah atas atau bagi bahwa baik banyak baru bawah beberapa begitu belum
benar berapa bersama besar bila bisa boleh buat bukan dalam dan dapat dari dengan dia dulu hal hampir hanya harus
hingga ia ialah ini itu jadi jika juga justru kalau kami kamu karena kata ke kecil kembali kemudian kenapa kepada
ketika kita lagi lain lainnya lalu lama lebih maka masih masing melalui mana mau mereka merupakan meski mungkin
namun nanti nya oleh pada para paling per perlu pernah pula pun saat saja salah sama sambil sampai sangat saya
sebab sebagai sebagaimana sebelum sebuah secara sedang sehingga sejak sekali seluruh semua sendiri seorang seperti
sering serta sesuai setelah setiap siapa suatu sudah supaya tak tanpa tapi telah tentang tersebut tetap tetapi
tiap tidak untuk walau yaitu yakni yang dll dsb dst yg dgn utk tdk dr pd krn
""".split()
STOPWORD_EN = """
the and of to in is for on with as by an be this that it from at or are was were has have had not but which also
its into than then there their they them these those been being will would can could should may might all any
some such only other more most very about after before over under between per via
""".split()
bc_stop = sc.broadcast(set(STOPWORD_ID) | set(STOPWORD_EN))

POLA_URL  = re.compile(r"https?://\S+|www\.\S+")
POLA_KATA = re.compile(r"[^\W\d_]+")

def tokenisasi(teks):
    teks = POLA_URL.sub(" ", teks.lower())
    return [k for k in POLA_KATA.findall(teks) if len(k) >= 3]

def cetak_top(hasil, judul, kolom, total):
    print(judul)
    print(f"{'#':>3}  {kolom:<52}{'jumlah':>8}{'porsi':>8}")
    for i, (k, n) in enumerate(hasil, 1):
        print(f"{i:>3}  {str(k)[:52]:<52}{n:>8,}{n / total:>8.1%}")

TAHAP = {}     # catatan jumlah task per tahap untuk ringkasan
print(f"Stopword: {len(bc_stop.value)} kata | contoh:", tokenisasi("/* Sejarah */ Mengembalikan suntingan [[Jakarta]] 2026"))

Stopword: 211 kata | contoh: ['sejarah', 'mengembalikan', 'suntingan', 'jakarta']


## Sel 4. (i) 10 wiki dengan edit manusia terbanyak
**Tahap 0 (narrow):** `textFile` → `flatMap(parse)` → `filter(type=edit, bot=false)` → `map((wiki, 1))`
**Tahap 1 (wide):** `reduceByKey(add, 4)` → `takeOrdered(10)`

In [4]:
# ===== SEL 4: (i) Top 10 wiki edit manusia =====
if "i" not in PERTANYAAN:
    print("Dilewati: pertanyaan (i) dikerjakan driver lain.")
else:
    tahap0_i = pantau(parsed.filter(lambda r: r[TYPE] == "edit" and not r[BOT])      # narrow
                            .map(lambda r: (r[WIKI], 1)),                              # narrow
                      f"{D} 2b-i tahap 0: filter+map (narrow)")
    tahap1_i = pantau(tahap0_i.reduceByKey(add, N_REDUCE),                             # WIDE
                      f"{D} 2b-i tahap 1: reduceByKey (wide)")

    tandai(f"{D}-2b-i Top10 wiki edit manusia")
    t = time.time()
    top_i = tahap1_i.takeOrdered(10, key=lambda kv: (-kv[1], kv[0]))                  # action
    dt_i = time.time() - t
    TAHAP["(i)"] = (tahap0_i.getNumPartitions(), tahap1_i.getNumPartitions())

    tandai(f"{D}-2b-i-cek Total edit manusia")
    total_i = parsed.filter(lambda r: r[TYPE] == "edit" and not r[BOT]).count()
    cetak_top(top_i, "\n(i) 10 wiki dengan edit manusia terbanyak", "wiki", total_i)
    print(f"\nTotal edit manusia: {total_i:,} (pembanding: 128,869) ->", "COCOK" if total_i == 128_869 else "BEDA")
    print(f"Task tahap 0: {TAHAP['(i)'][0]} | tahap 1: {TAHAP['(i)'][1]} | waktu {dt_i:.1f} dtk (termasuk jeda pantau)")

Dilewati: pertanyaan (i) dikerjakan driver lain.


## Sel 5. (ii) 10 halaman paling banyak disunting di idwiki
**Tahap 0 (narrow):** `textFile` → `flatMap(parse)` → `filter(wiki=idwiki)` → `map((title, 1))`
**Tahap 1 (wide):** `reduceByKey(add, 4)` → `takeOrdered(10)`. Seri diurutkan alfabetis.

In [5]:
# ===== SEL 5: (ii) Top 10 halaman idwiki =====
if "ii" not in PERTANYAAN:
    print("Dilewati: pertanyaan (ii) dikerjakan driver lain.")
else:
    tahap0_ii = pantau(parsed.filter(lambda r: r[WIKI] == "idwiki")                    # narrow
                             .map(lambda r: (r[TITLE], 1)),                            # narrow
                       f"{D} 2b-ii tahap 0: filter+map (narrow)")
    tahap1_ii = pantau(tahap0_ii.reduceByKey(add, N_REDUCE),                           # WIDE
                       f"{D} 2b-ii tahap 1: reduceByKey (wide)")

    tandai(f"{D}-2b-ii Top10 halaman idwiki")
    t = time.time()
    top_ii = tahap1_ii.takeOrdered(10, key=lambda kv: (-kv[1], kv[0]))                # action
    dt_ii = time.time() - t
    TAHAP["(ii)"] = (tahap0_ii.getNumPartitions(), tahap1_ii.getNumPartitions())

    tandai(f"{D}-2b-ii-cek Total suntingan idwiki")
    total_ii = parsed.filter(lambda r: r[WIKI] == "idwiki").count()
    cetak_top(top_ii, "\n(ii) 10 halaman idwiki paling banyak disunting", "judul halaman", total_ii)
    print(f"\nTotal suntingan idwiki: {total_ii:,} (pembanding: 372) ->", "COCOK" if total_ii == 372 else "BEDA")
    print(f"Task tahap 0: {TAHAP['(ii)'][0]} | tahap 1: {TAHAP['(ii)'][1]} | waktu {dt_ii:.1f} dtk (termasuk jeda pantau)")


(ii) 10 halaman idwiki paling banyak disunting
  #  judul halaman                                         jumlah   porsi
  1  Universitas Indonesia                                      6    1.6%
  2  Wikipedia:Bak pasir                                        6    1.6%
  3  Ghanta Ghar, Multan                                        5    1.3%
  4  Liga B Liga Negara UEFA 2026–2027                          4    1.1%
  5  Pengguna:PeterJunParker/Bak pasir                          4    1.1%
  6  Regio Patalis                                              4    1.1%
  7  Abryna                                                     3    0.8%
  8  Liga A Liga Negara UEFA 2026–2027                          3    0.8%
  9  Monyet rhesus                                              3    0.8%
 10  Pertempuran Radzymin (1920)                                3    0.8%

Total suntingan idwiki: 372 (pembanding: 372) -> COCOK
Task tahap 0: 8 | tahap 1: 4 | waktu 399.9 dtk (termasuk jeda pantau)


## Sel 6. (iii) 15 kata terbanyak pada comment idwiki
**Tahap 0 (narrow):** `textFile` → `flatMap(parse)` → `filter(idwiki & comment)` → `flatMap(tokenisasi)` → `filter(stopword, broadcast)` → `map((kata, 1))`
**Tahap 1 (wide):** `reduceByKey(add, 4)` → `takeOrdered(15)`

In [6]:
# ===== SEL 6: (iii) Top 15 kata comment idwiki =====
if "iii" not in PERTANYAAN:
    print("Dilewati: pertanyaan (iii) dikerjakan driver lain.")
else:
    tahap0_iii = pantau(parsed.filter(lambda r: r[WIKI] == "idwiki" and r[COMMENT])   # narrow
                              .flatMap(lambda r: tokenisasi(r[COMMENT]))              # narrow
                              .filter(lambda k: k not in bc_stop.value)               # narrow
                              .map(lambda k: (k, 1)),                                 # narrow
                        f"{D} 2b-iii tahap 0: filter+flatMap+filter+map (narrow)")
    tahap1_iii = pantau(tahap0_iii.reduceByKey(add, N_REDUCE),                         # WIDE
                        f"{D} 2b-iii tahap 1: reduceByKey (wide)")

    tandai(f"{D}-2b-iii Top15 kata comment idwiki")
    t = time.time()
    top_iii = tahap1_iii.takeOrdered(15, key=lambda kv: (-kv[1], kv[0]))              # action
    dt_iii = time.time() - t
    TAHAP["(iii)"] = (tahap0_iii.getNumPartitions(), tahap1_iii.getNumPartitions())

    tandai(f"{D}-2b-iii-cek Statistik token")
    komentar = parsed.filter(lambda r: r[WIKI] == "idwiki" and r[COMMENT])
    n_komentar = komentar.count()
    n_token = komentar.flatMap(lambda r: tokenisasi(r[COMMENT])).count()
    n_bersih = komentar.flatMap(lambda r: tokenisasi(r[COMMENT])).filter(lambda k: k not in bc_stop.value).count()
    cetak_top(top_iii, "\n(iii) 15 kata terbanyak pada comment idwiki", "kata", n_bersih)
    print(f"\nComment idwiki dipakai: {n_komentar:,} (pembanding: 286) ->", "COCOK" if n_komentar == 286 else "BEDA")
    print(f"Token: {n_token:,} -> setelah stopword: {n_bersih:,} ({1 - n_bersih / max(n_token, 1):.0%} terbuang)")
    print(f"Task tahap 0: {TAHAP['(iii)'][0]} | tahap 1: {TAHAP['(iii)'][1]} | waktu {dt_iii:.1f} dtk (termasuk jeda pantau)")

Dilewati: pertanyaan (iii) dikerjakan driver lain.


## Sel 7. Ringkasan 2b: narrow/wide, stage, dan task

In [7]:
# ===== SEL 7: Ringkasan 2b =====
RANTAI = {
    "(i)":   ["textFile", "flatMap parse", "filter edit&manusia", "map (wiki,1)"],
    "(ii)":  ["textFile", "flatMap parse", "filter idwiki", "map (title,1)"],
    "(iii)": ["textFile", "flatMap parse", "filter idwiki&comment", "flatMap tokenisasi",
              "filter stopword", "map (kata,1)"],
}
TOP = {"(i)": "takeOrdered(10)", "(ii)": "takeOrdered(10)", "(iii)": "takeOrdered(15)"}
print(f"{'Soal':<6}{'Tahap 0 (narrow)':<88}{'task':>5}  {'Tahap 1 (wide)':<32}{'task':>5}")
for q in RANTAI:
    if q.strip("()") not in PERTANYAAN:
        continue
    t0, t1 = TAHAP.get(q, ("-", "-"))
    print(f"{q:<6}{' -> '.join(RANTAI[q]):<88}{t0:>5}  {'reduceByKey -> ' + TOP[q]:<32}{t1:>5}")
print("\nSetiap soal: 1 job (dipicu takeOrdered), 2 stage, dipisahkan shuffle reduceByKey.")
print("Accumulator baris rusak (snapshot asli):", acc_rusak.value, "-> harus 0 (snapshot asli tidak berisi baris rusak)")
print("Spark UI:", sc.uiWebUrl, f"-> Jobs -> label {D}-2b-... -> DAG Visualization")

Soal  Tahap 0 (narrow)                                                                         task  Tahap 1 (wide)                   task
(ii)  textFile -> flatMap parse -> filter idwiki -> map (title,1)                                 8  reduceByKey -> takeOrdered(10)      4

Setiap soal: 1 job (dipicu takeOrdered), 2 stage, dipisahkan shuffle reduceByKey.
Accumulator baris rusak (snapshot asli): 0 -> harus 0 (snapshot asli tidak berisi baris rusak)
Spark UI: http://192.168.1.15:4041 -> Jobs -> label D2-2b-... -> DAG Visualization


## Sel 8. Simpan hasil driver ini
Hasil disimpan di folder `hasil_2b` (di samping notebook, PC Master) agar driver 4 bisa mencocokkannya.

In [8]:
# ===== SEL 8: Simpan hasil =====
HASIL_DIR = Path("hasil_2b"); HASIL_DIR.mkdir(exist_ok=True)
hasil = {"driver": NOMOR_DRIVER, "app_id": sc.applicationId, "waktu": time.strftime("%Y-%m-%d %H:%M:%S")}
if "i" in PERTANYAAN:   hasil["i"] = top_i
if "ii" in PERTANYAAN:  hasil["ii"] = top_ii
if "iii" in PERTANYAAN: hasil["iii"] = top_iii
f = HASIL_DIR / f"driver{NOMOR_DRIVER}.json"
f.write_text(json.dumps(hasil, ensure_ascii=False, indent=1), encoding="utf-8")
print("Disimpan:", f.resolve())

Disimpan: C:\BigData\notebooks\hasil_2b\driver2.json


## Sel 9 (driver 4). Cocokkan hasil dengan driver 1–3
Jalankan setelah driver 1, 2, dan 3 menjalankan Sel 8.

In [9]:
# ===== SEL 9: Driver 4 mencocokkan hasil =====
if NOMOR_DRIVER != 4:
    print("Sel ini hanya untuk driver 4.")
else:
    for q, n in (("i", 1), ("ii", 2), ("iii", 3)):
        f = HASIL_DIR / f"driver{n}.json"
        if not f.exists():
            print(f"({q}) driver {n}: belum ada {f.name}"); continue
        lain = [tuple(x) for x in json.loads(f.read_text(encoding="utf-8"))[q]]
        sendiri = [tuple(x) for x in hasil[q]]
        print(f"({q}) driver {n} vs driver 4:", "SAMA" if lain == sendiri else "BERBEDA")
        if lain != sendiri:
            for a, b in zip(lain, sendiri):
                print("   ", a, "|", b)

Sel ini hanya untuk driver 4.


## Sel 10. Penutup
Selama aplikasi hidup, Spark UI driver ini bisa dibuka. Setelah screenshot selesai, hentikan agar core dilepas.

In [ ]:
# ===== SEL 10: Penutup =====
print("Spark UI:", sc.uiWebUrl)
# spark.stop()   # buka komentar setelah screenshot selesai, agar 4 core driver ini dilepas